# 01 · Data pipeline
**Owner:** Adrian · Group 20 · Dutch Housing Blame

**Research question.** Who do people blame for the Dutch housing crisis online, and do Dutch-language
(r/thenetherlands) and English-language (r/Netherlands) communities blame different actors?

**Goal of this notebook.** Turn the raw weekly Arctic Shift files into clean, pseudonymised housing data that the rest of the group can use.

**Input:** `data/raw/{posts,comments}/<subreddit>/<week>.jsonl.zst` (from `collect.py`)
**Output:** `comments_housing.parquet`, `posts.parquet`, `author_activity.parquet` in `data/processed/`, plus descriptive figures.

**Done when:**
- [ ] all check cells pass
- [ ] processed files match the contract in the guide
- [ ] processed files uploaded to SURFdrive

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 20
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
ANNOTATION = ROOT / "annotation"
FIGURES = ROOT / "figures"
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

In [ ]:
import gc
import hashlib
import html
import io
import os
import re
import zipfile

import zstandard
from dotenv import load_dotenv
from lingua import Language, LanguageDetectorBuilder

SUBREDDITS = ["thenetherlands", "Netherlands"]
DATA_PROC.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(exist_ok=True)

## Data collection
The data was collected with `collect.py` from the [Arctic Shift API](https://github.com/ArthurHeitmann/arctic_shift),
an archive of public Reddit data. Arctic Shift's keyword search timed out on these two large subreddits, so we
downloaded **everything** posted between 2024-01-01 and 2026-10-31 and filter for housing ourselves (task 4).

- The period is split into 148 weeks (UTC). For each week, subreddit and type (posts/comments) the script asks for all items
  in that week, oldest first, page by page, until a page comes back with fewer than 100 items.
- Each finished week is saved as one compressed file, so the crawl can be stopped and restarted without losing work.
- It sends at most one request per second; on rate limits or timeouts it waits and retries, and weeks that keep
  failing are logged and fetched again on a second run (2 weeks needed this).

Our counts compared with Arctic Shift's own counts (`/api/time_series`):

| | collected | Arctic Shift | coverage |
|---|---|---|---|
| r/thenetherlands posts | 38,913 | 38,931 | 99.95% |
| r/Netherlands posts | 79,747 | 79,782 | 99.96% |
| r/thenetherlands comments | 1,419,203 | 1,419,457 | 99.98% |
| r/Netherlands comments | 1,701,447 | 1,701,326 | 100.0% |

## Task 1 · Load the raw data
**Why (for the paper):** The data section reports how much we collected and that it is complete.

**1A. Reading**
- A reusable way to read one week file (zstd-compressed JSON lines).
- All posts in `posts_raw` and all comments in `comments_raw`, with only the columns we need.
- 3 million comments is a lot of memory – think about how you build the DataFrame.

In [ ]:
POST_COLS = ["id", "author", "created_utc", "subreddit", "score", "title", "selftext", "num_comments"]
COMMENT_COLS = ["id", "author", "created_utc", "subreddit", "score", "body", "link_id", "parent_id"]
TEXT_COLS = ["id", "author", "subreddit", "title", "selftext", "body", "link_id", "parent_id"]


def read_week(path):
    with open(path, "rb") as f:
        lines = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(f), encoding="utf-8")
        return [json.loads(line) for line in lines]


def load_kind(kind, columns):
    frames = []
    for sub in SUBREDDITS:
        files = sorted((DATA_RAW / kind / sub).glob("*.jsonl.zst"))
        print(f"{kind}/{sub}: {len(files)} files")
        for path in files:
            df = pd.DataFrame(read_week(path), columns=columns)
            # pyarrow strings use much less memory than Python objects
            frames.append(df.astype({c: "string[pyarrow]" for c in columns if c in TEXT_COLS}))
    return pd.concat(frames, ignore_index=True)

In [ ]:
posts_raw = load_kind("posts", POST_COLS)
comments_raw = load_kind("comments", COMMENT_COLS)
print(f"comments_raw: {comments_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")

We read one file at a time and convert the text columns to pyarrow strings before concatenating; with normal
Python strings the 3.1 million comments did not fit comfortably in memory.

**1B. Completeness**
- `raw_counts`: a dict with the number of rows per (kind, subreddit), e.g. `{('posts', 'thenetherlands'): 38913, ...}`.
- 148 files per folder – check it.

In [ ]:
raw_counts = {(kind, sub): int((df["subreddit"] == sub).sum())
              for kind, df in [("posts", posts_raw), ("comments", comments_raw)]
              for sub in SUBREDDITS}
raw_counts

In [ ]:
# check (do not change)
EXPECTED = {("posts", "thenetherlands"): 38913, ("posts", "Netherlands"): 79747,
            ("comments", "thenetherlands"): 1419203, ("comments", "Netherlands"): 1701447}
assert raw_counts == EXPECTED, raw_counts
print("ok")

**Go further (optional):**
- Comments per week per subreddit – any unusual weeks?
- Can you halve the memory use of `comments_raw`?

**Interpretation:** All 592 week files load, and the counts match the crawl exactly (38,913 + 79,747 posts, 1,419,203 + 1,701,447 comments). Converting the text columns to pyarrow strings per file keeps the 3.1 million comments at about 1 GB in memory, so the whole pipeline runs on a normal laptop in a few minutes.

## Task 2 · Clean
**Why (for the paper):** Deleted content differs a lot between the subreddits; this goes into the data section and the limitations.

**2A. Drop what we can't use**
- Deleted, removed, empty and bot comments are dropped.
- `dropped`: a table with the number dropped per reason and subreddit.

In [ ]:
BOT = r"(?i)^automoderator$|bot$"


def drop_reason(df):
    text = df["body"].fillna("")
    reason = pd.Series(pd.NA, index=df.index, dtype="string")
    reason[text.str.strip() == ""] = "empty"
    reason[text == "[deleted]"] = "deleted"
    reason[text == "[removed]"] = "removed"
    reason[df["author"].fillna("").str.contains(BOT)] = "bot"
    return reason


comments_raw["drop_reason"] = drop_reason(comments_raw)
dropped = comments_raw.groupby("subreddit")["drop_reason"].value_counts().unstack(fill_value=0)
dropped["total"] = comments_raw.groupby("subreddit").size()
dropped["kept"] = dropped["total"] - dropped[["bot", "deleted", "empty", "removed"]].sum(axis=1)
dropped["dropped_share"] = (1 - dropped["kept"] / dropped["total"]).round(3)
dropped = dropped.reset_index()
dropped

Bots are AutoModerator and accounts whose name ends in "bot" (RemindMeBot etc.). We keep comments whose *account*
was deleted but whose text is still there; they get no `author_hash` (task 3).

**2B. Clean the text**
- Comment `text` and post titles without URLs, markdown link syntax, HTML entities or extra whitespace – test your cleaning on a few tricky examples first.
- `n_words` per comment.

In [ ]:
MD_LINK = re.compile(r"\[([^\]]*)\]\([^)]*\)")
URL = re.compile(r"https?://\S+|www\.\S+")
USER = re.compile(r"(?<![\w/])/?u/[A-Za-z0-9_-]+")
MARKDOWN = re.compile(r"[*_~`#^]+|^\s*>", re.M)
ESCAPED = re.compile(r"\\([^\w\s])")


def clean_text(text):
    text = MD_LINK.sub(r"\1", text)
    text = URL.sub(" ", text)
    text = html.unescape(text)
    text = ESCAPED.sub(r"\1", text)
    text = USER.sub("u/[user]", text)
    text = MARKDOWN.sub(" ", text)
    return " ".join(text.split())


tests = {
    "Zie [dit artikel](https://nos.nl/x) over **huurprijzen**": "Zie dit artikel over huurprijzen",
    "&gt; landlords &amp; investors\n\nexactly": "landlords & investors exactly",
    "\>eigen woning forfait weg": "eigen woning forfait weg",
    "ask u/some_user or /u/Other-1 about https://example.com/a?b=c": "ask u/[user] or u/[user] about",
}
for raw, expected in tests.items():
    assert clean_text(raw) == expected, (clean_text(raw), expected)
print("clean_text ok")

In [ ]:
comments = comments_raw[comments_raw["drop_reason"].isna()].copy()
comments["text"] = comments["body"].map(clean_text).astype("string[pyarrow]")
comments["n_words"] = comments["text"].str.count(r"\S+").astype(int)

posts = posts_raw.copy()
posts["title"] = posts["title"].fillna("").map(clean_text).astype("string[pyarrow]")
posts["removed"] = posts["selftext"].isin(["[removed]", "[deleted]"])
print(len(comments), "comments kept")

In [ ]:
# check (do not change)
assert "subreddit" in dropped.columns or "subreddit" in dropped.index.names
print(dropped)

**Go further (optional):**
- Deleted/removed share per month per subreddit (plot).
- Usernames mentioned inside the text (`u/...`) replaced as well. ✓ (done in `clean_text`)

**Interpretation:** r/thenetherlands loses 9.5% of its comments and r/Netherlands only 2.3%. Almost all of that is content *removed* (by moderators or Reddit), not deleted by the author: 8.8% vs. 1.9%. This is a real difference between the communities and a limitation: if removed comments are more often blaming comments, r/thenetherlands' blame is under-represented. We clean URLs, markdown and mentioned usernames before matching keywords, so a word inside a link does not count as a housing comment.

## Task 3 · Pseudonymise
**Why (for the paper):** Privacy & ethics is 4 points in the paper rubric – we must describe exactly how we protect users.

**3A. Hashing**
- Every username becomes `author_hash`: the first 16 characters of a salted SHA-256 hash.
- The salt lives in a `.env` file (`PSEUDONYM_SALT`), never in the notebook or in git, and is shared privately with the group.

In [ ]:
load_dotenv(ROOT / ".env")
SALT = os.environ["PSEUDONYM_SALT"]


def pseudonymise(name):
    if pd.isna(name) or name == "[deleted]":
        return None
    return hashlib.sha256((SALT + name).encode("utf-8")).hexdigest()[:16]


def hash_column(names):
    # hash every unique name once instead of 3 million times
    lookup = {name: pseudonymise(name) for name in pd.unique(names)}
    return names.map(lookup).astype("string[pyarrow]")


assert pseudonymise("example") == pseudonymise("example") != pseudonymise("Example")
comments_raw["author_hash"] = hash_column(comments_raw["author"])
posts["author_hash"] = hash_column(posts["author"])
comments["author_hash"] = comments_raw.loc[comments.index, "author_hash"]

**3B. Parents and clean-up**
- `parent_author_hash`: who wrote the comment or post that this comment replies to (needs *all* comments and posts, not only housing ones).
- No raw username left anywhere. The result is called `comments`.

In [ ]:
author_of = pd.concat([
    pd.Series(comments_raw["author_hash"].to_numpy(), index="t1_" + comments_raw["id"].astype(str)),
    pd.Series(posts["author_hash"].to_numpy(), index="t3_" + posts["id"].astype(str)),
])
comments["parent_author_hash"] = comments["parent_id"].astype(str).map(author_of).astype("string[pyarrow]")

comments = comments.drop(columns=["author", "body", "drop_reason"])
posts = posts.drop(columns=["author", "selftext"])
del comments_raw, posts_raw, author_of
gc.collect()
print("parent author known for", f"{comments['parent_author_hash'].notna().mean():.1%}", "of comments")

In [ ]:
# check (do not change)
assert "author" not in comments.columns
assert comments["author_hash"].dropna().str.len().eq(16).all()
print("ok")

**Go further (optional):**
- Two sentences for the paper: why a salt, and why it stays out of git.

Without a salt anyone could hash a known username and look it up in our data; with a secret salt the hashes can only be
linked to usernames by someone who has the salt. Comments by deleted accounts have no `author_hash`.

**Interpretation:** Every username is replaced by a salted hash, computed once per unique name. For 99.3% of the comments we also know (as a hash) who they reply to, which notebook 04 needs for the reply network. Comments whose account was deleted keep their text but get no hash, so they can be analysed but not linked to a user.

## Task 4 · Housing filter
**Why (for the paper):** A filter that catches non-housing comments weakens every result; the precision goes into the method section.

**4A. Keywords**
- A bilingual keyword list (Dutch + English). Start from the 12-week test list and decide as a group about `hypotheek`/`mortgage`.
- Dutch compounds (huurwoningen) must match too.

In [ ]:
NL_TERMS = [
    "woningnood", "woningcrisis", "wooncrisis", "woningmarkt", "woningtekort", "woningzoekend", "woningbouw",
    "huurwoning", "huurhuis", "huurhuizen", "huurprijs", "huurprijzen", "huurverhoging", "huurbescherming",
    "huurcommissie", "huurtoeslag", "huurder", "verhuurder", "huisjesmelker", "woningcorporatie", "sociale huur",
    "middenhuur", "vrije sector", "scheefwon", "overbied", "opkoopbescherming", "betaalbare huur",
    "starterswoning", "koopwoning", "huizenprijs", "huizenprijzen", "huizenmarkt", "hypotheekrenteaftrek",
    "kamernood", "studentenhuisvesting", "huisvesting",
]
EN_TERMS = [
    "housing", "rental", "rent price", "rent control", "rent increase", "landlord", "tenant", "house prices",
    "property prices", "outbid", "overbid", "mortgage interest deduction",
]
# \b only on the left: "huurwoning" also matches "huurwoningen", "landlord" also "landlords"
HOUSING_PATTERN = r"\b(?:" + "|".join(re.escape(t) for t in NL_TERMS + EN_TERMS) + ")"

**Decision on `hypotheek` / `mortgage`:** left out. In the 12-week test they were the most frequent match in
r/thenetherlands (about a third of all matches), and most of those comments are personal finance ("my mortgage
application"), not the housing debate. The policy term *hypotheekrenteaftrek* / *mortgage interest deduction* stays in.

**4B. Flags**
- `kw_match` on comments, `housing_title` on posts, `in_housing_thread` on comments whose post has a housing title.
- `comments_housing`: comments with `kw_match` or `in_housing_thread`, with counts per subreddit.

In [ ]:
comments["kw_match"] = comments["text"].str.contains(HOUSING_PATTERN, case=False, regex=True)
posts["housing_title"] = posts["title"].str.contains(HOUSING_PATTERN, case=False, regex=True)

housing_posts = set("t3_" + posts.loc[posts["housing_title"], "id"].astype(str))
comments["in_housing_thread"] = comments["link_id"].isin(housing_posts)

comments_housing = comments[comments["kw_match"] | comments["in_housing_thread"]].copy()
pd.DataFrame({
    "comments": comments.groupby("subreddit").size(),
    "kw_match": comments.groupby("subreddit")["kw_match"].sum(),
    "in_housing_thread": comments.groupby("subreddit")["in_housing_thread"].sum(),
    "housing (either)": comments_housing.groupby("subreddit").size(),
    "housing posts": posts.groupby("subreddit")["housing_title"].sum(),
})

**4C. Precision**
- `precision_sample`: 100 random `kw_match` comments that you labelled by hand (`is_housing` 1/0).
- Precision overall and per subreddit, a look at the false positives, and what you changed because of them.

The sample is exported once to `data/processed/precision_sample.csv`. Open it, fill `is_housing` with 1 (about housing)
or 0 (not about housing) for every row, save, and run the cells below again.

In [ ]:
PRECISION_FILE = DATA_PROC / "precision_sample.csv"
if not PRECISION_FILE.exists():
    (comments_housing[comments_housing["kw_match"]]
     .sample(100, random_state=SEED)[["id", "subreddit", "text"]]
     .assign(is_housing="")
     .to_csv(PRECISION_FILE, index=False))
    print("Exported. Label the is_housing column in", PRECISION_FILE)

precision_sample = pd.read_csv(PRECISION_FILE)
if precision_sample["is_housing"].notna().all():
    print("precision overall:", round(precision_sample["is_housing"].mean(), 3))
    print(precision_sample.groupby("subreddit")["is_housing"].agg(["mean", "size"]))
else:
    print(precision_sample["is_housing"].isna().sum(), "rows still need a label")

In [ ]:
# check (do not change)
assert len(precision_sample) == 100 and precision_sample["is_housing"].notna().all()
print("precision:", precision_sample["is_housing"].mean())

**Go further (optional):**
- Which terms cause most false positives?
- Precision of comments that are only `in_housing_thread`.

**Interpretation:** _Fill in after labelling the precision sample._ 113,592 comments count as housing-related: 74,792 in r/Netherlands and 38,800 in r/thenetherlands. A housing keyword appears in 2.6% of r/Netherlands comments vs. 1.6% in r/thenetherlands, and r/Netherlands has almost four times as many housing posts (2,170 vs. 595), so housing is a bigger topic for the international community. About half of the housing comments are only included because they are in a housing thread.

## Task 5 · Context and language
**Why (for the paper):** Annotators need context, and since the communities share many users we also compare by language.

**5A. Parent context**
- `parent_text`: the parent comment's text, or the post title for top-level comments (empty if the parent is missing).
- How many parents are missing, and why?

In [ ]:
text_of = pd.concat([
    pd.Series(comments["text"].to_numpy(), index="t1_" + comments["id"].astype(str)),
    pd.Series(posts["title"].to_numpy(), index="t3_" + posts["id"].astype(str)),
])
parent_text = comments_housing["parent_id"].astype(str).map(text_of)
missing = parent_text.isna().groupby(comments_housing["parent_id"].str[:3]).mean().rename("missing share")
comments_housing["parent_text"] = parent_text.fillna("").astype("string[pyarrow]")
missing

**5B. Language**
- `lang` with `lingua`: `nl`, `en`, `mixed` or `other`. Your own rule for `mixed` and for very short texts – explain it.
- A manual check: does it look right on a random sample per language?

Lingua's confidence for a whole text hides mixing (a half-Dutch, half-English sentence still gets 99% Dutch), so we let
lingua split each comment into language segments and count the words per language:
- fewer than 3 words → `other` (too short to tell);
- at least 3 Dutch *and* 3 English words, and the smaller of the two is at least 25% of the words → `mixed`;
- otherwise the language with at least half of the words (`nl` or `en`), else `other`.

In [ ]:
detector = LanguageDetectorBuilder.from_languages(
    Language.DUTCH, Language.ENGLISH, Language.GERMAN, Language.FRENCH).build()


def lang_from_segments(segments):
    words = {}
    for s in segments:
        words[s.language] = words.get(s.language, 0) + s.word_count
    total = sum(words.values())
    nl, en = words.get(Language.DUTCH, 0), words.get(Language.ENGLISH, 0)
    if total < 3:
        return "other"
    if min(nl, en) >= 3 and min(nl, en) / total >= 0.25:
        return "mixed"
    if nl / total >= 0.5:
        return "nl"
    if en / total >= 0.5:
        return "en"
    return "other"


examples = {
    "Welke sociale huur? In de grote steden zit die potdicht.": "nl",
    "There is a housing crisis, so you would certainly struggle.": "en",
    "De huurprijzen zijn echt belachelijk. This landlord is insane and the government does nothing.": "mixed",
    "Dit is echt gezellig": "nl",
    "lol": "other",
}
for text, expected in examples.items():
    got = lang_from_segments(detector.detect_multiple_languages_of(text))
    print(f"{got:6} {expected:6} {text}")

In [ ]:
segments = detector.detect_multiple_languages_in_parallel_of(comments_housing["text"].tolist())
comments_housing["lang"] = [lang_from_segments(s) for s in segments]
del segments

In [ ]:
for lang in ["nl", "en", "mixed", "other"]:
    print(f"--- {lang}")
    for text in comments_housing.loc[comments_housing["lang"] == lang, "text"].sample(5, random_state=SEED):
        print("  ", text[:120])

In [ ]:
# check (do not change)
assert set(comments_housing["lang"].unique()) <= {"nl", "en", "mixed", "other"}
print(comments_housing.groupby("subreddit")["lang"].value_counts(normalize=True).round(3))

**Go further (optional):**
- Share of English in r/thenetherlands and Dutch in r/Netherlands per quarter.

**Interpretation:** Only 0.3% of the housing comments have a missing parent (deleted or posted before 2024). The language matches the community almost perfectly: 94.8% of r/thenetherlands housing comments are Dutch and 95.6% of r/Netherlands comments are English, so for housing the subreddit and the language are nearly the same split. *Mixed* is rare (<1%) and includes English comments with a few Dutch housing terms (\"makelaar\"), so we treat it as an upper bound.

## Task 6 · User activity and descriptives
**Why (for the paper):** Notebook 04 needs the activity table; the data section needs a figure that shows the data at a glance.

**6A. User activity**
- `author_activity`: per user and subreddit the number of comments (all comments), housing comments, first and last date.
- Sanity check: roughly 76k users in r/thenetherlands and 168k in r/Netherlands.

In [ ]:
comments["created"] = pd.to_datetime(comments["created_utc"], unit="s", utc=True)
comments_housing["created"] = pd.to_datetime(comments_housing["created_utc"], unit="s", utc=True)
posts["created"] = pd.to_datetime(posts["created_utc"], unit="s", utc=True)

author_activity = (comments.dropna(subset=["author_hash"])
                   .groupby(["author_hash", "subreddit"])
                   .agg(n_comments=("id", "size"), first_seen=("created", "min"), last_seen=("created", "max"))
                   .reset_index())
housing_counts = (comments_housing.dropna(subset=["author_hash"])
                  .groupby(["author_hash", "subreddit"]).size().rename("n_housing_comments").reset_index())
author_activity = author_activity.merge(housing_counts, on=["author_hash", "subreddit"], how="left")
author_activity["n_housing_comments"] = author_activity["n_housing_comments"].fillna(0).astype(int)
author_activity["author_hash"] = author_activity["author_hash"].astype(str)
author_activity.groupby("subreddit")["author_hash"].nunique()

**6B. Figures**
- Housing comments per month per subreddit, as counts and as a share of all comments.
- Language mix per subreddit.
- Saved as `figures/01_<name>.png` (`dpi=300`).

In [ ]:
def per_month(df):
    month = df["created"].dt.tz_convert(None).dt.to_period("M").dt.to_timestamp()
    return df.groupby([month, "subreddit"]).size().unstack()


housing_month = per_month(comments_housing)
share_month = housing_month / per_month(comments)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
housing_month.plot(ax=axes[0])
axes[0].set_ylabel("housing comments")
(share_month * 100).plot(ax=axes[1], legend=False)
axes[1].set_ylabel("% of all comments")
axes[1].set_xlabel("")
axes[0].set_title("Housing comments per month")
fig.tight_layout()
fig.savefig(FIGURES / "01_housing_per_month.png", dpi=300, bbox_inches="tight")

In [ ]:
lang_mix = pd.crosstab(comments_housing["subreddit"], comments_housing["lang"], normalize="index") * 100
ax = lang_mix[["nl", "en", "mixed", "other"]].plot.barh(stacked=True, figsize=(8, 2.8))
ax.set_xlabel("% of housing comments")
ax.set_ylabel("")
ax.legend(ncol=4, bbox_to_anchor=(0.5, -0.35), loc="upper center")
ax.set_title("Language of housing comments")
plt.savefig(FIGURES / "01_language_mix.png", dpi=300, bbox_inches="tight")
lang_mix.round(1)

**Go further (optional):**
- Top 20 housing terms per subreddit (word counts, like in the A1 notebook).

In [ ]:
terms = (comments_housing.loc[comments_housing["kw_match"], ["subreddit", "text"]]
         .assign(term=lambda d: d["text"].str.lower().str.findall(HOUSING_PATTERN.replace("(?:", "(", 1)))
         .explode("term").dropna(subset=["term"]))
terms.groupby("subreddit")["term"].value_counts().groupby(level=0).head(10)

**Interpretation:** The activity table has 76,531 users in r/thenetherlands and 167,863 in r/Netherlands, the same as the first check on the raw data. The figures show the monthly volume of housing comments and its share of all comments, plus the language mix used in the data section.

## Save outputs
- `comments_housing`, `posts` and `author_activity` saved as parquet in `data/processed/`, with the columns from the guide.
- Uploaded to the SURFdrive processed folder, and the group knows.

In [ ]:
HOUSING_COLS = ["id", "link_id", "parent_id", "subreddit", "created", "author_hash", "parent_author_hash",
                "score", "text", "parent_text", "kw_match", "in_housing_thread", "lang", "n_words"]
POST_OUT = ["id", "subreddit", "created", "author_hash", "title", "score", "num_comments", "removed", "housing_title"]

comments_housing[HOUSING_COLS].to_parquet(DATA_PROC / "comments_housing.parquet", index=False)
posts[POST_OUT].to_parquet(DATA_PROC / "posts.parquet", index=False)
author_activity.to_parquet(DATA_PROC / "author_activity.parquet", index=False)

# one zip to share with the group (pseudonymised, no usernames)
with zipfile.ZipFile(DATA_PROC / "group20_processed.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["comments_housing.parquet", "posts.parquet", "author_activity.parquet"]:
        z.write(DATA_PROC / name, arcname=name)
for f in sorted(DATA_PROC.glob("*")):
    print(f"{f.name:28} {f.stat().st_size / 1e6:7.1f} MB")

In [ ]:
# check (do not change)
REQUIRED = ["id", "link_id", "parent_id", "subreddit", "created", "author_hash", "parent_author_hash",
            "score", "text", "parent_text", "kw_match", "in_housing_thread", "lang", "n_words"]
df = pd.read_parquet(DATA_PROC / "comments_housing.parquet")
missing = set(REQUIRED) - set(df.columns)
assert not missing, missing
assert not {"author", "parent_author"} & set(df.columns)
for f in ["posts.parquet", "author_activity.parquet"]:
    assert (DATA_PROC / f).exists(), f
print(len(df), "housing comments, ok")

## Notes for the paper
- Table: comments left after each step.
- Precision of the housing filter and the hypotheek/mortgage decision.
- How usernames are pseudonymised.